← [Las otras dos señales](09-las-otras-dos-senales.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Cómo se mide](11-como-se-mide.ipynb) →

# 10 · Votación y decisión

Seis diagnósticos, una compuerta. Este documento explica cómo se pasa de lo uno
a lo otro, y por qué la política es la que es.



## Estado vigente del proyecto (actualizado el 13 de agosto de 2026)

Esta serie conserva explicaciones y resultados históricos, pero la referencia
operativa actual es la siguiente:

- El sistema experto tiene **193 reglas**, CF estilo MYCIN, meta-reglas,
  encadenamiento hacia adelante y hacia atrás. Su voto usa OpenAI como
  proveedor principal, con heurísticas OpenCV que refinan atributos.
- El modelo local que participa en la decisión es **MobileNetV2 TFLite
  float32**, corrida `run_20260721_2129`; clasifica solo `plastico | vidrio`.
  MobileNetV3-Large INT8 está archivado como respaldo y **no emite votos**.
- En 1.000 capturas OV3660/QVGA, V2 obtuvo **71,60 %** de exactitud y
  **71,25 %** de macro-F1; V3 INT8 obtuvo 57,10 % y 57,09 %. La validación
  histórica de 98,43 % no describe por sí sola el rendimiento del robot.
- La ESP32-CAM toma tres fotos: se suman los seis votos válidos de ambas
  fuentes. `desconocido` es abstención; un empate se resuelve con el proveedor.
  Si el proveedor se abstiene las tres veces, el modelo local necesita 3/3.

La documentación operativa es [`ia/vision-service/README.md`](../../ia/vision-service/README.md),
[`model/README.md`](../../ia/vision-service/model/README.md) y
[`PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md`](../../docs/PLAN-INTEGRACION-VISION-MAIN-2026-08-13.md).


## Por qué votar y no promediar

La tentación obvia sería promediar las confianzas de ambas señales. **No se
puede**, y la razón está en el documento [04](04-factor-de-certeza.ipynb):

| | Sistema experto | Red neuronal |
| --- | --- | --- |
| Qué produce | Factor de certeza MYCIN | Probabilidad softmax |
| Suma sobre hipótesis | No suma nada en particular | Siempre 1,0 |
| Origen | Juicio del experto | Reparto interno de la red |

Un CF de 0,95 y una softmax de 0,95 **no son la misma magnitud**. Promediarlos
produciría un número sin significado. Por eso `voting.py` cuenta votos en vez de
mezclar confianzas: la confianza viaja como diagnóstico visible, pero no pondera
la decisión.



## Las reglas de la votación actual

La cámara toma tres fotos. Cada una aporta un diagnóstico de
`openai_sistema_experto` y otro de `modelo_local`. Para decidir deben llegar los
**tres diagnósticos de cada fuente**; una respuesta incompleta devuelve
`desconocido`.

`desconocido` es una abstención: no suma a ningún material. Las confianzas se
conservan como diagnóstico, pero no se promedian.

1. Se suman todos los votos válidos de ambas fuentes.
2. Gana la clase con más votos totales.
3. Si hay empate, decide la preferencia de los votos válidos de
   OpenAI+sistema experto.
4. Si el proveedor se abstuvo tres veces, el modelo local solo autoriza con
   unanimidad 3/3; una mayoría 2–1 devuelve `desconocido`.
5. Si falta una respuesta o el empate no se resuelve, no se abre compuerta.

| Proveedor | Modelo local | Resultado |
| --- | --- | --- |
| P, P, D | P, V, P | plástico, 4–1 |
| V, V, D | V, P, V | vidrio, 4–1 |
| D, D, D | V, V, V | vidrio, unanimidad local |
| D, D, D | V, V, P | desconocido |
| P, P, V | V, V, P | plástico, 3–3; desempata el proveedor |

`P` = plástico, `V` = vidrio y `D` = desconocido/abstención.


## Por qué el modelo local sigue siendo una señal vigilada

El modelo local solo conoce `plastico | vidrio`; frente a una lata, cartón o una mano siempre devuelve una de esas dos clases. El proveedor+sistema experto puede abstenerse o reconocer categorías que no abren compuerta.

La política conjunta aprovecha la señal local, pero no convierte una mayoría local 2–1 en apertura cuando el proveedor se abstuvo tres veces. Sobre 1.000 capturas OV3660/QVGA, MobileNetV2 acertó 716/1.000 (71,60 %) y la mayoría por tripletas 248/330 (75,15 %). Esto bastó para elegirlo frente a V3 INT8, no para afirmar que el clasificador binario pueda rechazar residuos ajenos.


## Tres fotos, no una

La ESP32-CAM captura una ráfaga de tres por residuo. La idea es que un encuadre
malo o un reflejo puntual no arruine la decisión.

Conviene ser honesto sobre su límite: en 32 ráfagas del mismo objeto, el modelo
local dio el mismo resultado en 30 (93,8 %). Es **estable en el acierto y en el
error**. Tres fotos ayudan contra el ruido momentáneo, no contra un error
sistemático de dominio ([08](08-cambio-de-dominio.ipynb)).



## Qué pasa si el servicio no responde

Una decisión que conviene conocer porque afecta al robot en producción. Si el
servicio de visión está caído, sin cuota o inalcanzable, la ruta de Next.js
**no propaga el error**:

```json
{ "material": "desconocido", "confidence": 0, "rule_applied": "servicio de visión no disponible — ..." }
```

Devuelve una abstención. El robot no abre nada y muestra el mensaje
correspondiente. Es la misma política conservadora: ante la duda —incluso si la
duda es un fallo de infraestructura— rechazar antes que abrir la compuerta
equivocada.



## El principio de fondo

Todo el sistema está construido sobre una asimetría de costos:

> Abrir la compuerta equivocada contamina un contenedor entero. No abrir ninguna
> solo molesta a una persona.

Por eso la abstención es barata y el error es caro, y por eso hay umbrales,
mayorías estrictas y jerarquías en vez de un simple "gana el más confiado".

---

← [Las otras dos señales](09-las-otras-dos-senales.ipynb) · [Índice](00-indice.ipynb) · Siguiente: [Cómo se mide](11-como-se-mide.ipynb) →
